# 06 — Intervention D: sliced inference (SAHI)

Inference only, on the **baseline weights** (notebook 02) — no retraining. Each native 1920×1080 frame is tiled, every tile is detected at imgsz 640, and tile detections are merged with a full-frame pass by NMS. Tiles see the ball at native resolution instead of 3× downscaled.

- **Slice size is chosen on val** (640 or 960 native px, by ball mAP50-95), then reported on test once.
- **Same metric code as every other run.** SAHI boxes are scored by `bc.evaluate_predictions`, which reuses ultralytics' own matching and `ap_per_class`. The first check proves it: plain full-frame predictions through that evaluator must reproduce `model.val()` on test.
- Scale caveat for the report: the baseline was trained on 3× downscaled frames, so objects in native tiles look larger than anything it saw in training (scale jitter covers only 0.5–1.5×). That mismatch is part of what this measures.
- FPS is timed at the normal confidence threshold 0.25; scoring uses 0.001 like `model.val()`.

In [ ]:
SMOKE = False  # True: only the first slice size, written to <run_name>_smoke
BASELINE_WEIGHTS = "/content/results/baseline_yolo11s_640/best.pt"
SLICE_SIZES = [640, 960]
OVERLAP = 0.2

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sahi"], check=True)
sys.path.insert(0, "/content/ballhawk")
import pandas as pd
import ballhawk_common as bc
from ultralytics import YOLO

data_yaml = bc.prepare_data()
if SMOKE:
    SLICE_SIZES = SLICE_SIZES[:1]

In [ ]:
# Evaluator check: full-frame predictions through bc.evaluate_predictions vs model.val() on test.
model = YOLO(BASELINE_WEIGHTS)
def full_frame(img):
    b = model.predict(img, imgsz=640, conf=0.001, iou=0.7, max_det=300, verbose=False)[0].boxes
    return b.xyxy.cpu().numpy(), b.conf.cpu().numpy(), b.cls.cpu().numpy()
ours, ours_pc = bc.evaluate_predictions({p: full_frame(p) for p in bc.split_images(data_yaml, "test")}, data_yaml, "test")
ref, ref_pc = bc.evaluate_test(BASELINE_WEIGHTS, data_yaml, 640)
print("evaluator:", ours, "\nmodel.val:", ref)
assert abs(ours["mAP50"] - ref["mAP50"]) < 0.02, "evaluator does not reproduce model.val(); SAHI numbers would not be comparable"

In [ ]:
def ball(per_class):
    return next(c for c in per_class if c["class"] == "ball")

score_dm = bc.sahi_model(BASELINE_WEIGHTS, 0.001)
sweep = []
for size in SLICE_SIZES:
    ov, pc = bc.evaluate_predictions({p: bc.sahi_arrays(score_dm, p, size, OVERLAP) for p in bc.split_images(data_yaml, "val")},
                                     data_yaml, "val")
    sweep.append({"slice": size, "val_mAP50_95": ov["mAP50_95"], "val_ball_mAP50_95": ball(pc)["mAP50_95"]})
sweep = pd.DataFrame(sweep); sweep

In [ ]:
best = int(sweep.loc[sweep.val_ball_mAP50_95.idxmax(), "slice"])
overall, per_class = bc.evaluate_predictions(
    {p: bc.sahi_arrays(score_dm, p, best, OVERLAP) for p in bc.split_images(data_yaml, "test")}, data_yaml, "test")
time_dm = bc.sahi_model(BASELINE_WEIGHTS, 0.25)
fps = bc.measure_fps(None, data_yaml, 640, predict=lambda p: bc.sahi_predict(time_dm, p, best, OVERLAP))

run_name = f"sahi_baseline_s{best}" + ("_smoke" if SMOKE else "")
record = {
    "run_name": run_name, "notebook": "06",
    "variable_changed": f"sliced inference: {best}px native slices, overlap {OVERLAP}, + full-frame pass, NMS merge",
    "model": "baseline_yolo11s_640 best.pt", "imgsz": 640, "epochs": 100, "batch": 16, "split": "test",
    "overall": overall, "per_class": per_class, "inference_fps": fps, "train_minutes": 0.0,
    "notes": (f"Inference only on 02 weights. Slice size chosen on val from {SLICE_SIZES} by ball mAP50-95. "
              f"Evaluator check on test, full-frame: bc.evaluate_predictions mAP50 {ours['mAP50']:.4f} vs "
              f"model.val {ref['mAP50']:.4f}. ultralytics 8.4.163."),
}
print(bc.write_metrics(record, bc.RESULTS_DIR / run_name))

In [ ]:
pd.DataFrame({"full_frame": {c["class"]: c["mAP50_95"] for c in ours_pc},
              f"sahi_{best}": {c["class"]: c["mAP50_95"] for c in per_class}}).round(3)